# Baseline Faster R-CNN v2

## Objetivo

Ajustar (`fine-tuning`) Faster R-CNN v2 con backbone ResNet-50 FPN a las ocho especies de parásitos. Partimos de pesos preentrenados en COCO, entrenamos con `train`, usamos `val` para seguir el experimento y seleccionar el mejor checkpoint, y reservamos `test` para una única evaluación final. Este notebook es el primer pipeline experimental; no afirma resultados hasta que se ejecute el entrenamiento.

Las imágenes y anotaciones ya convertidas se leen desde `data/converted/parasites-1s07h/`. La entrada de torchvision usa imágenes RGB como tensores `[C, H, W]`, cajas `[x1, y1, x2, y2]` y etiquetas 1–8 (la etiqueta 0 queda reservada al fondo). El notebook de comprobación anterior explica el formato COCO y muestra las cajas reales.

## Configuración inicial

La configuración de abajo es un punto de partida explícito, no una decisión definitiva del grupo: 10 épocas, batch de 2 y SGD. Hay que registrar los valores realmente usados y no comparar resultados de ejecuciones con configuraciones distintas como si fueran el mismo baseline. El modelo aplica su propio resize/normalización estándar; aquí no añadimos aumentos de datos.

Por seguridad, `RUN_TRAINING` y `RUN_FINAL_TEST` empiezan en `False`: ejecutar todas las celdas no inicia accidentalmente un entrenamiento largo ni descarga pesos. Cuando se quiera iniciar el fine-tuning, revisar la configuración, poner `RUN_TRAINING = True` y ejecutar las celdas en orden. Los pesos/checkpoints se guardan en `models/` (local, ignorado por Git); las métricas ligeras van a `results/metrics/`.

## 1. Imports y configuración del experimento

`DEVICE` elige CUDA si PyTorch la detecta y, si no, CPU. `NUM_CLASSES` incluye el fondo además de las ocho categorías. La semilla ayuda a repetir el muestreo y la inicialización, aunque no garantiza determinismo bit a bit en todos los dispositivos.

In [ ]:
import json
import random
import time
from collections import defaultdict
from datetime import datetime, timezone
from functools import partial

import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import clear_output, display
from PIL import Image
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from torch.utils.data import DataLoader
from torchvision.models.detection import (
    FasterRCNN_ResNet50_FPN_V2_Weights,
    fasterrcnn_resnet50_fpn_v2,
)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.transforms.functional import to_tensor
from tqdm.auto import tqdm

from parasite_detection.paths import DATA, MODELS, RESULTS

SEED = 42
BATCH_SIZE = 2
NUM_EPOCHS = 10
EARLY_STOPPING_PATIENCE = 3
LEARNING_RATE = 0.005
MOMENTUM = 0.9
WEIGHT_DECAY = 0.0005
NUM_WORKERS = 0
RUN_TRAINING = True
RUN_FINAL_TEST = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
WEIGHTS = FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT
CHECKPOINT_PATH = MODELS / "faster_rcnn_v2_resnet50_fpn_baseline_best.pth"
HISTORY_PATH = RESULTS / "metrics" / "faster_rcnn_v2_baseline_history.json"
TEST_METRICS_PATH = RESULTS / "metrics" / "faster_rcnn_v2_baseline_test.json"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Dispositivo: {DEVICE}")
print(f"Dataset convertido: {DATA}")
print(f"Pesos iniciales: {WEIGHTS}")
print(f"Configuración: {NUM_EPOCHS} épocas, batch={BATCH_SIZE}, SGD lr={LEARNING_RATE}")
print(f"Early stopping: {EARLY_STOPPING_PATIENCE} épocas consecutivas de aumento de val_loss")
assert (DATA / "annotations" / "instances_train.json").is_file(), (
    "Faltan datos convertidos. Ejecuta scripts/download_dataset.py y "
    "scripts/prepare_dataset.py desde la raíz del proyecto."
)

## 2. Cargador COCO y `DataLoader`

Estas funciones son la versión de trabajo del cargador validado en el notebook anterior. Leen una imagen cada vez, agrupan anotaciones por `image_id` y convierten `[x, y, ancho, alto]` a `[x1, y1, x2, y2]`. El `collate_fn` mantiene imágenes y objetivos como listas porque detección admite distinto número de cajas por imagen (y no debe apilarlas a ciegas).

No usamos una clase personalizada: el `DataLoader` itera sobre índices y `collate_fn` carga las muestras correspondientes.

In [2]:
def load_coco_split(split, data_dir=DATA):
    """Carga las imágenes y anotaciones COCO de train, val o test."""
    if split not in {"train", "val", "test"}:
        raise ValueError(f"Partición no reconocida: {split}")

    annotation_path = data_dir / "annotations" / f"instances_{split}.json"
    image_dir = data_dir / "images" / split
    with annotation_path.open(encoding="utf-8") as file:
        coco = json.load(file)

    image_records = coco["images"]
    categories = {category["id"]: category["name"] for category in coco["categories"]}
    annotations_by_image = defaultdict(list)
    for annotation in coco["annotations"]:
        annotations_by_image[annotation["image_id"]].append(annotation)

    return image_records, categories, annotations_by_image, image_dir, annotation_path


def load_detection_sample(index, image_records, annotations_by_image, image_dir):
    """Devuelve un tensor RGB y el target que espera torchvision."""
    image_info = image_records[index]
    image_id = image_info["id"]
    with Image.open(image_dir / image_info["file_name"]) as file:
        image = to_tensor(file.convert("RGB"))

    boxes, labels, areas, iscrowd = [], [], [], []
    for annotation in annotations_by_image[image_id]:
        x, y, width, height = annotation["bbox"]
        if width <= 0 or height <= 0:
            raise ValueError(f"Caja no válida en imagen {image_id}: {annotation['bbox']}")
        boxes.append([x, y, x + width, y + height])
        labels.append(annotation["category_id"])
        areas.append(annotation["area"])
        iscrowd.append(annotation["iscrowd"])

    target = {
        "boxes": torch.as_tensor(boxes, dtype=torch.float32).reshape(-1, 4),
        "labels": torch.as_tensor(labels, dtype=torch.int64),
        "image_id": torch.tensor([image_id], dtype=torch.int64),
        "area": torch.as_tensor(areas, dtype=torch.float32),
        "iscrowd": torch.as_tensor(iscrowd, dtype=torch.int64),
    }
    return image, target


def collate_detection_batch(indices, image_records, annotations_by_image, image_dir):
    """Carga un batch y conserva imágenes/targets como tuplas de muestras."""
    samples = [
        load_detection_sample(index, image_records, annotations_by_image, image_dir)
        for index in indices
    ]
    return tuple(zip(*samples))


def make_detection_loader(split, batch_size, shuffle):
    """Crea un DataLoader para una partición COCO."""
    records, categories, annotations, image_dir, annotation_path = load_coco_split(split)
    collate_fn = partial(
        collate_detection_batch,
        image_records=records,
        annotations_by_image=annotations,
        image_dir=image_dir,
    )
    loader = DataLoader(
        range(len(records)),
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        collate_fn=collate_fn,
    )
    return loader, records, categories, annotation_path


train_loader, train_records, categories, _ = make_detection_loader(
    "train", BATCH_SIZE, shuffle=True
)
val_loader, val_records, val_categories, val_annotation_path = make_detection_loader(
    "val", BATCH_SIZE, shuffle=False
)
assert categories == val_categories
NUM_CLASSES = len(categories) + 1  # Las etiquetas 1–8 son especies; 0 es background.
print(f"Train: {len(train_records)} imágenes | Val: {len(val_records)} imágenes")
print(f"Categorías: {categories}")
print(f"Salidas de clasificación, incluido background: {NUM_CLASSES}")

Train: 1484 imágenes | Val: 215 imágenes
Categorías: {1: 'Ancylostoma Spp', 2: 'Ascaris Lumbricoides', 3: 'Enterobius Vermicularis', 4: 'Fasciola Hepatica', 5: 'Hymenolepis', 6: 'Schistosoma', 7: 'Taenia Sp', 8: 'Trichuris Trichiura'}
Salidas de clasificación, incluido background: 9


## 3. Crear el modelo preentrenado

`fasterrcnn_resnet50_fpn_v2` parte de pesos COCO. Conservamos el backbone/FPN preentrenados y sustituimos el predictor de cajas para que clasifique las ocho especies más el fondo (9 salidas). No cargamos los Parquet en el modelo: el modelo recibe tensores de imagen y diccionarios target del cargador anterior.

In [3]:
def create_model(num_classes, device):
    """Construye Faster R-CNN v2 preentrenado con una cabeza de num_classes."""
    model = fasterrcnn_resnet50_fpn_v2(weights=WEIGHTS)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model.to(device)


def move_targets_to_device(targets, device):
    """Mueve cada tensor del target al dispositivo del modelo."""
    return [
        {key: value.to(device) if torch.is_tensor(value) else value for key, value in target.items()}
        for target in targets
    ]

## 4. Entrenamiento y evaluación COCO

Cada época optimiza la suma de pérdidas que devuelve el detector en modo entrenamiento. Después calculamos también la pérdida de validación (sin gradientes) y mAP/AP50 COCO. La gráfica de pérdidas se actualiza al terminar cada época; el mejor checkpoint se sigue escogiendo por mAP de validación.

En inferencia Faster R-CNN ya devuelve cajas en `[x1, y1, x2, y2]`; para COCO las convertimos de vuelta a `[x, y, ancho, alto]`. Los `category_id` de las predicciones se mantienen en 1–8.

In [4]:
def train_one_epoch(model, loader, optimizer, device):
    """Ejecuta una época de fine-tuning y devuelve la pérdida media."""
    model.train()
    running_loss = 0.0
    for images, targets in tqdm(loader, desc="Entrenamiento", leave=False):
        images = [image.to(device) for image in images]
        targets = move_targets_to_device(targets, device)

        loss_dict = model(images, targets)
        loss = sum(loss_dict.values())
        if not torch.isfinite(loss):
            details = {name: float(value.detach().cpu()) for name, value in loss_dict.items()}
            raise FloatingPointError(f"Pérdida no finita: {details}")

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        running_loss += float(loss.detach().cpu())

    return running_loss / len(loader)


def evaluate_loss(model, loader, device):
    """Calcula la pérdida media en modo entrenamiento, sin actualizar pesos."""
    model.train()
    running_loss = 0.0
    with torch.no_grad():
        for images, targets in tqdm(loader, desc="Pérdida de validación", leave=False):
            images = [image.to(device) for image in images]
            targets = move_targets_to_device(targets, device)
            loss_dict = model(images, targets)
            loss = sum(loss_dict.values())
            if not torch.isfinite(loss):
                details = {name: float(value.detach().cpu()) for name, value in loss_dict.items()}
                raise FloatingPointError(f"Pérdida de validación no finita: {details}")
            running_loss += float(loss.detach().cpu())

    return running_loss / len(loader)


def plot_losses(epoch_results):
    """Actualiza la gráfica de pérdidas de train y validación en el notebook."""
    clear_output(wait=True)
    epochs = [result["epoch"] for result in epoch_results]
    train_losses = [result["train_loss"] for result in epoch_results]
    validation_losses = [result["validation_loss"] for result in epoch_results]

    figure, axis = plt.subplots(figsize=(8, 5))
    axis.plot(epochs, train_losses, marker="o", label="Training")
    axis.plot(epochs, validation_losses, marker="o", label="Validación")
    axis.set(title="Pérdida por época", xlabel="Época", ylabel="Pérdida")
    axis.set_xticks(epochs)
    axis.legend()
    axis.grid(True, alpha=0.3)
    figure.tight_layout()
    display(figure)
    plt.close(figure)


def evaluate_coco(model, loader, annotation_path, device):
    """Evalúa predicciones con COCO mAP y AP50 en el split indicado."""
    coco_gt = COCO(str(annotation_path))
    predictions = []
    model.eval()
    with torch.inference_mode():
        for images, targets in tqdm(loader, desc="Evaluación", leave=False):
            outputs = model([image.to(device) for image in images])
            for target, output in zip(targets, outputs):
                image_id = int(target["image_id"].item())
                boxes = output["boxes"].detach().cpu().tolist()
                scores = output["scores"].detach().cpu().tolist()
                labels = output["labels"].detach().cpu().tolist()
                for box, score, label in zip(boxes, scores, labels):
                    x1, y1, x2, y2 = box
                    predictions.append({
                        "image_id": image_id,
                        "category_id": int(label),
                        "bbox": [x1, y1, x2 - x1, y2 - y1],
                        "score": float(score),
                    })

    if not predictions:
        raise RuntimeError("El modelo no produjo predicciones; no se puede calcular COCO mAP.")

    coco_dt = coco_gt.loadRes(predictions)
    evaluator = COCOeval(coco_gt, coco_dt, iouType="bbox")
    evaluator.params.imgIds = coco_gt.getImgIds()
    evaluator.evaluate()
    evaluator.accumulate()
    evaluator.summarize()
    return {"map": float(evaluator.stats[0]), "map_50": float(evaluator.stats[1])}


def experiment_config():
    """Devuelve los ajustes que se registrarán con los resultados."""
    return {
        "model": "torchvision fasterrcnn_resnet50_fpn_v2",
        "initial_weights": str(WEIGHTS),
        "num_classes_including_background": NUM_CLASSES,
        "seed": SEED,
        "batch_size": BATCH_SIZE,
        "epochs": NUM_EPOCHS,
        "early_stopping": {
            "monitor": "validation_loss",
            "patience": EARLY_STOPPING_PATIENCE,
            "rule": "stop after consecutive epoch-over-epoch increases",
        },
        "optimizer": "SGD",
        "learning_rate": LEARNING_RATE,
        "momentum": MOMENTUM,
        "weight_decay": WEIGHT_DECAY,
        "device": str(DEVICE),
        "train_images": len(train_records),
        "validation_images": len(val_records),
        "selection_metric": "validation COCO mAP (IoU 0.50:0.95)",
    }

## 5. Ejecutar el fine-tuning

Antes de cambiar `RUN_TRAINING` a `True`, confirma que `DEVICE` es el esperado y que los valores de configuración son adecuados para la memoria/tiempo disponible. `EARLY_STOPPING_PATIENCE` fija cuántas épocas consecutivas debe aumentar `validation_loss` para detener el entrenamiento. La gráfica se actualiza al terminar cada época, el historial se guarda después de cada época y el checkpoint solo se actualiza cuando mejora el mAP de validación.

In [ ]:
if RUN_TRAINING:
    MODELS.mkdir(parents=True, exist_ok=True)
    HISTORY_PATH.parent.mkdir(parents=True, exist_ok=True)
    model = create_model(NUM_CLASSES, DEVICE)
    trainable_parameters = [parameter for parameter in model.parameters() if parameter.requires_grad]
    optimizer = torch.optim.SGD(
        trainable_parameters,
        lr=LEARNING_RATE,
        momentum=MOMENTUM,
        weight_decay=WEIGHT_DECAY,
    )

    config = experiment_config()
    history = {
        "status": "in_progress",
        "started_at": datetime.now(timezone.utc).isoformat(),
        "config": config,
        "epochs": [],
    }
    best_map = float("-inf")
    previous_val_loss = None
    consecutive_val_loss_increases = 0

    for epoch in range(1, NUM_EPOCHS + 1):
        epoch_started = time.perf_counter()
        train_loss = train_one_epoch(model, train_loader, optimizer, DEVICE)
        val_loss = evaluate_loss(model, val_loader, DEVICE)
        val_metrics = evaluate_coco(model, val_loader, val_annotation_path, DEVICE)
        if previous_val_loss is not None and val_loss > previous_val_loss:
            consecutive_val_loss_increases += 1
        else:
            consecutive_val_loss_increases = 0
        previous_val_loss = val_loss
        epoch_result = {
            "epoch": epoch,
            "train_loss": train_loss,
            "validation_loss": val_loss,
            "consecutive_validation_loss_increases": consecutive_val_loss_increases,
            "validation_map": val_metrics["map"],
            "validation_map_50": val_metrics["map_50"],
            "duration_seconds": time.perf_counter() - epoch_started,
        }
        history["epochs"].append(epoch_result)
        plot_losses(history["epochs"])
        print(json.dumps(epoch_result, indent=2))

        if val_metrics["map"] > best_map:
            best_map = val_metrics["map"]
            torch.save({
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "epoch": epoch,
                "validation_metrics": val_metrics,
                "config": config,
            }, CHECKPOINT_PATH)
            history["best_checkpoint"] = str(CHECKPOINT_PATH)
            print(f"Mejor checkpoint guardado: {CHECKPOINT_PATH}")

        if consecutive_val_loss_increases >= EARLY_STOPPING_PATIENCE:
            history["status"] = "early_stopped"
            history["stop_reason"] = (
                f"validation_loss aumentó durante {EARLY_STOPPING_PATIENCE} épocas consecutivas"
            )
            print(f"Early stopping: {history['stop_reason']}")

        HISTORY_PATH.write_text(json.dumps(history, indent=2) + "\n", encoding="utf-8")
        if history["status"] == "early_stopped":
            break

    if history["status"] == "in_progress":
        history["status"] = "completed"
    history["completed_at"] = datetime.now(timezone.utc).isoformat()
    HISTORY_PATH.write_text(json.dumps(history, indent=2) + "\n", encoding="utf-8")
    print(f"Entrenamiento finalizado ({history['status']}). Historial: {HISTORY_PATH}")
else:
    print("Entrenamiento desactivado. Revisa la configuración y cambia RUN_TRAINING a True para iniciarlo.")

## 6. Evaluación final en test

Esta sección se ejecuta por separado y solo después de seleccionar el mejor checkpoint usando `val`. No se utiliza `test` para elegir época ni hiperparámetros. Al habilitarla, carga el checkpoint local y escribe las métricas finales; los resultados solo existen tras ejecutar correctamente la celda.

In [ ]:
if RUN_FINAL_TEST:
    if not CHECKPOINT_PATH.is_file():
        raise FileNotFoundError(
            f"No existe el checkpoint {CHECKPOINT_PATH}. Ejecuta primero el fine-tuning."
        )

    test_loader, test_records, test_categories, test_annotation_path = make_detection_loader(
        "test", BATCH_SIZE, shuffle=False
    )
    assert test_categories == categories
    test_model = create_model(NUM_CLASSES, DEVICE)
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=True)
    test_model.load_state_dict(checkpoint["model_state_dict"])
    test_metrics = evaluate_coco(test_model, test_loader, test_annotation_path, DEVICE)

    test_result = {
        "status": "completed",
        "evaluated_at": datetime.now(timezone.utc).isoformat(),
        "checkpoint_epoch": checkpoint["epoch"],
        "test_images": len(test_records),
        "test_metrics": test_metrics,
        "config": checkpoint["config"],
    }
    TEST_METRICS_PATH.parent.mkdir(parents=True, exist_ok=True)
    TEST_METRICS_PATH.write_text(json.dumps(test_result, indent=2) + "\n", encoding="utf-8")
    print(json.dumps(test_result, indent=2))
    print(f"Métricas finales guardadas: {TEST_METRICS_PATH}")
else:
    print("Evaluación final desactivada. Actívala solo después de seleccionar el checkpoint con val.")

Evaluación final desactivada. Actívala solo después de seleccionar el checkpoint con val.


## Qué registrar y siguiente paso

El historial identifica el estado como `in_progress`, `completed` o `early_stopped`, y guarda la configuración junto a las pérdidas de entrenamiento/validación, los aumentos consecutivos de `validation_loss`, mAP/AP50 de validación, duración por época y mejor checkpoint. El mecanismo para cuando `validation_loss` aumenta durante el número de épocas indicado por `EARLY_STOPPING_PATIENCE`; el checkpoint se sigue seleccionando por mAP. La evaluación final en test se guarda en otro JSON para no confundirla con validación.

No cambies parámetros después de ver test. Si se ajusta la configuración durante el desarrollo, crea una nueva ejecución con otro nombre de historial/checkpoint o registra claramente que es otra corrida; no sobrescribas resultados que quieras conservar. Cuando este notebook esté validado y el baseline se haya ejecutado, las funciones reutilizables se pueden trasladar a `src/parasite_detection/` y el entrenamiento a un script.